# 03 - Compare and Score

Primeira versao jogavel: conecta moveset, video de referencia, webcam, comparacao, score, feedback e HUD. Toda a logica reutilizavel permanece em `core/`.

In [ ]:
MOVESET_PATH = "C:/Users/dvmrn/Repositórios/ApenasDance-AiMotionTrackingDanceGame/macarena_moveset.json"
VIDEO_PATH = "C:/Users/dvmrn/Repositórios/ApenasDance-AiMotionTrackingDanceGame/macarena_pose.mp4"

In [ ]:
from __future__ import annotations

import importlib
import time
from pathlib import Path

import cv2
import numpy as np

import core.comparison_engine as comparison_engine_module
import core.final_score as final_score_module
import core.video_player as video_player_module

importlib.reload(comparison_engine_module)
importlib.reload(final_score_module)
importlib.reload(video_player_module)

from core.camera_capture import CameraCapture
from core.comparison_engine import ComparisonEngine
from core.feedback import FeedbackMapper
from core.final_score import FinalScoreCalculator, GameResult
from core.joint_angle_calculator import JointAngleCalculator
from core.pose_comparator import PoseComparator
from core.pose_detector import PoseDetector
from core.pose_normalizer import PoseNormalizer
from core.pose_source import MovesetPoseSource, RealtimePoseSource
from core.video_player import VideoPlayer
from core.visualizers.comparison_hud import ComparisonHUD

## Score hibrido

A engine usa 70% de similaridade entre landmarks normalizados e 30% de similaridade entre angulos articulares. Landmarks capturam a forma global do corpo, enquanto angulos ajudam a manter a avaliacao robusta para articulacoes especificas.

In [ ]:
class DanceComparison:
    """Orchestrate the playable dance comparison flow."""

    def __init__(
        self,
        moveset_path: str | Path = MOVESET_PATH,
        video_path: str | Path = VIDEO_PATH,
        camera_index: int = 0,
        model_path: str | Path | None = None,
    ) -> None:
        self.moveset_path = moveset_path
        self.video_path = video_path
        self.camera_index = camera_index
        self.model_path = model_path

        self.camera = CameraCapture(camera_index=self.camera_index)
        self.detector = PoseDetector(model_path=self.model_path)
        self.video_player = VideoPlayer(self.video_path)
        self.normalizer = PoseNormalizer()
        self.angle_calculator = JointAngleCalculator()
        self.moveset_source = MovesetPoseSource(
            moveset_path=self.moveset_path,
            normalizer=self.normalizer,
            angle_calculator=self.angle_calculator,
        )
        self.realtime_source: RealtimePoseSource | None = None
        self.engine = ComparisonEngine(
            moveset_source=self.moveset_source,
            comparator=PoseComparator(),
            feedback_mapper=FeedbackMapper(),
        )
        self.hud = ComparisonHUD()
        self.final_score_calculator = FinalScoreCalculator()
        self.game_result: GameResult | None = None
        self.fps = 0.0
        self._last_tick = time.perf_counter()
        self._quit_requested = False

    def run(self) -> None:
        """Run countdown, playback, real-time comparison and final screen."""
        self._initialize()
        try:
            if self._run_countdown() and not self._quit_requested:
                self._start_gameplay()
                completed = self._run_game_loop()
                if completed:
                    self.game_result = self._compute_final_result()
                    self._show_final_result(self.game_result)
        finally:
            self._close_gameplay_hud()
            self._release_resources()

    def _initialize(self) -> None:
        """Open webcam, detector and reference video."""
        self.camera.open()
        self.detector.initialize()
        self.video_player.open()
        self.video_player.pause()

    def _run_countdown(self) -> bool:
        """Show webcam and paused reference video during countdown."""
        for text in self.video_player.countdown_sequence():
            started_at = time.perf_counter()
            duration = 0.75 if text == "JÁ!" else 1.0
            while time.perf_counter() - started_at < duration:
                success, webcam_frame = self.camera.read()
                _, reference_frame = self.video_player.read()
                if not success or webcam_frame is None or reference_frame is None:
                    raise RuntimeError("Could not render countdown frames.")

                countdown_view = self.hud.draw_countdown(reference_frame, webcam_frame, text, self.fps)
                self.hud.show(countdown_view)
                if self._should_quit():
                    self._quit_requested = True
                    return False
        return True

    def _start_gameplay(self) -> None:
        """Align realtime pose timestamps with reference video playback."""
        self.realtime_source = RealtimePoseSource(
            camera=self.camera,
            detector=self.detector,
            normalizer=self.normalizer,
            angle_calculator=self.angle_calculator,
        )
        self.video_player.reset()
        self.video_player.play()
        self._last_tick = time.perf_counter()

    def _run_game_loop(self) -> bool:
        """Process realtime poses and draw the comparison HUD."""
        if self.realtime_source is None:
            raise RuntimeError("Realtime pose source is not initialized.")

        while True:
            if self.video_player.is_finished:
                return True
            realtime_pose = self.realtime_source.get_next_pose()
            success, reference_frame = self.video_player.read()
            webcam_frame = self.realtime_source.latest_frame
            if not success or reference_frame is None or webcam_frame is None:
                return self.video_player.is_finished

            self._update_fps()
            result = self.engine.compare(realtime_pose)
            frame = self.hud.draw(
                reference_frame=reference_frame,
                webcam_frame=webcam_frame,
                realtime_pose=realtime_pose,
                result=result,
                fps=self.fps,
                score=self.engine.current_score(),
                feedback=result.feedback,
            )
            self.hud.show(frame)

            if self._should_quit():
                self._quit_requested = True
                return False

    def _compute_final_result(self) -> GameResult:
        """Aggregate final score and rank from existing project components."""
        duration_seconds = self.video_player.frame_count / self.video_player.fps
        return self.final_score_calculator.calculate(
            feedbacks=self.engine.feedback_history,
            similarity_average=self.engine.similarity_average(),
            duration_seconds=duration_seconds,
        )

    def _close_gameplay_hud(self) -> None:
        """Close only the gameplay HUD window before showing final result."""
        try:
            cv2.destroyWindow(self.hud.window_name)
        except cv2.error:
            pass

    def _show_final_result(self, result: GameResult) -> None:
        """Render the final score screen until Q or window close."""
        final_window_name = "Final Score"
        screen = np.full((520, 760, 3), 18, dtype=np.uint8)
        self._draw_centered_text(screen, "FIM DA DANÇA", 95, 1.6, (0, 255, 255), 3)
        self._draw_centered_text(screen, "Pontuação Final", 190, 1.1, (240, 240, 240), 2)
        self._draw_centered_text(screen, f"{result.final_score:.1f} / 100", 260, 1.45, (255, 255, 255), 3)
        self._draw_centered_text(screen, "Ranque", 350, 1.1, (240, 240, 240), 2)
        self._draw_centered_text(screen, result.ranque, 425, 2.0, (0, 255, 0), 4)

        cv2.namedWindow(final_window_name, cv2.WINDOW_AUTOSIZE)
        while True:
            cv2.imshow(final_window_name, screen)
            key = cv2.waitKey(50) & 0xFF
            try:
                window_visible = cv2.getWindowProperty(final_window_name, cv2.WND_PROP_VISIBLE) >= 1
            except cv2.error:
                window_visible = False
            if key == ord("q") or key == ord("Q") or not window_visible:
                break

    @staticmethod
    def _draw_centered_text(
        image: np.ndarray,
        text: str,
        y: int,
        scale: float,
        color: tuple[int, int, int],
        thickness: int,
    ) -> None:
        """Draw centered text on the final score screen."""
        font = cv2.FONT_HERSHEY_SIMPLEX
        size, _ = cv2.getTextSize(text, font, scale, thickness)
        x = (image.shape[1] - size[0]) // 2
        cv2.putText(image, text, (x, y), font, scale, color, thickness, cv2.LINE_AA)

    def _update_fps(self) -> None:
        """Update displayed FPS from wall-clock frame cadence."""
        current_tick = time.perf_counter()
        elapsed = current_tick - self._last_tick
        if elapsed > 0:
            current_fps = 1.0 / elapsed
            self.fps = current_fps if self.fps == 0.0 else (self.fps * 0.9) + (current_fps * 0.1)
        self._last_tick = current_tick

    @staticmethod
    def _should_quit() -> bool:
        """Return True when the user presses Q."""
        key = cv2.waitKey(1) & 0xFF
        return key == ord("q") or key == ord("Q")

    def _release_resources(self) -> None:
        """Release webcam, MediaPipe, video and OpenCV windows."""
        self.camera.release()
        self.detector.close()
        self.video_player.release()
        cv2.destroyAllWindows()

In [ ]:
game = DanceComparison()
game.run()